In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from google.colab import drive
import os

# 1. Montar Drive (se ainda não estiver montado)
drive.mount('/content/drive')

# 2. Configurações Globais
# Ajuste este caminho para onde seus arquivos .dat estão
DATASET_DIR = "/content/drive/MyDrive/2025/Estudos/Datasets/pamap2/"
OUTPUT_PATH = "/content/drive/MyDrive/2025/Estudos/Datasets/pamap2/pamap2_v2.npz"

# IDs dos sujeitos - Separando o 108 para Validação
TRAIN_SUBJECTS = [101, 102, 103, 104, 107]
VAL_SUBJECTS = [108]
TEST_SUBJECTS = [105, 106]

# Configuração da Janela Deslizante (PAMAP2 = 100Hz)
WINDOW_SIZE = 250  # 2.5 segundos
STEP_SIZE = 125    # 50% de overlap

# Definição das Colunas (Rótulos do PAMAP2)
COLUMNS = [
    "timestamp", "activityID", "heartrate",
    "handTemperature", "handAcc16_1", "handAcc16_2", "handAcc16_3",
    "handAcc6_1", "handAcc6_2", "handAcc6_3",
    "handGyro1", "handGyro2", "handGyro3",
    "handMagne1", "handMagne2", "handMagne3",
    "handOrientation1", "handOrientation2", "handOrientation3", "handOrientation4",
    "chestTemperature", "chestAcc16_1", "chestAcc16_2", "chestAcc16_3",
    "chestAcc6_1", "chestAcc6_2", "chestAcc6_3",
    "chestGyro1", "chestGyro2", "chestGyro3",
    "chestMagne1", "chestMagne2", "chestMagne3",
    "chestOrientation1", "chestOrientation2", "chestOrientation3", "chestOrientation4",
    "ankleTemperature", "ankleAcc16_1", "ankleAcc16_2", "ankleAcc16_3",
    "ankleAcc6_1", "ankleAcc6_2", "ankleAcc6_3",
    "ankleGyro1", "ankleGyro2", "ankleGyro3",
    "ankleMagne1", "ankleMagne2", "ankleMagne3",
    "ankleOrientation1", "ankleOrientation2", "ankleOrientation3", "ankleOrientation4"
]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
def load_and_clean_data(dataset_dir):
    print(" Carregando arquivos brutos...")
    data_collection = pd.DataFrame()

    # Iterar pelos arquivos dos sujeitos (subject101.dat a subject109.dat)
    # Nota: Vamos carregar todos que existirem na pasta
    for i in range(1, 10):
        subject_id = 100 + i
        filename = os.path.join(dataset_dir, f"subject{subject_id}.dat")

        if not os.path.exists(filename):
            print(f"    Aviso: Arquivo {filename} não encontrado. Pulando.")
            continue

        print(f"   Lendo subject{subject_id}.dat...")
        df = pd.read_table(filename, header=None, sep='\s+')
        df.columns = COLUMNS
        df["subject_id"] = subject_id # Adiciona ID do sujeito
        data_collection = pd.concat([data_collection, df], ignore_index=True)

    print("\n Limpando dados...")
    # 1. Remover colunas de orientação (ruidosas/desnecessárias)
    cols_to_drop = [c for c in data_collection.columns if 'Orientation' in c]
    df_clean = data_collection.drop(cols_to_drop, axis=1)

    # 2. Remover activityID 0 (dados de transição/sem label)
    df_clean = df_clean[df_clean["activityID"] != 0]

    # 3. Forçar numérico e interpolar dados faltantes (ex: heartrate tem NaN)
    # O PAMAP2 tem alguns NaNs nos sensores wireless. Interpolação linear resolve.
    df_clean = df_clean.apply(pd.to_numeric, errors='coerce')
    df_clean = df_clean.interpolate(method='linear', limit_direction='forward')
    df_clean = df_clean.dropna() # Remove o que não deu pra interpolar

    print(f" Dados carregados. Shape Total: {df_clean.shape}")
    return df_clean

def generate_windows(data, window_size, step_size):
    """
    Segmenta os dados em janelas deslizantes.
    """
    X_windows = []
    y_windows = []

    # Agrupa por (Sujeito, Atividade) para não misturar janelas
    # Ex: Não queremos uma janela que comece no sujeito 101 e termine no 102
    grouped = data.groupby(['subject_id', 'activityID'])

    for (subject, activity), group in grouped:
        # Garante ordem temporal
        group = group.sort_values(by='timestamp')

        # Remove colunas que não são features de sensor
        # (Timestamp, ActivityID e SubjectID não entram na rede neural)
        features = group.drop(['timestamp', 'activityID', 'subject_id'], axis=1).values
        label = activity

        # Sliding Window
        for i in range(0, len(group) - window_size + 1, step_size):
            window = features[i : i + window_size]
            X_windows.append(window)
            y_windows.append(label)

    return np.array(X_windows), np.array(y_windows)

<>:16: SyntaxWarning: invalid escape sequence '\s'
<>:16: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_10761/4255115791.py:16: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_table(filename, header=None, sep='\s+')


In [3]:
# 1. Carregar Tudo
df_all = load_and_clean_data(DATASET_DIR)

# 2. Divisão por Sujeitos (Protocolo Literatura)
print("\n Dividindo Treino, Validação e Teste (Subject Independent)...")
df_train = df_all[df_all['subject_id'].isin(TRAIN_SUBJECTS)].copy()
df_val = df_all[df_all['subject_id'].isin(VAL_SUBJECTS)].copy() # NEW
df_test = df_all[df_all['subject_id'].isin(TEST_SUBJECTS)].copy()

print(f"   Amostras Treino: {len(df_train)}")
print(f"   Amostras Validação: {len(df_val)}")
print(f"   Amostras Teste:  {len(df_test)}")

# 3. Normalização (Fit no Treino, Transform no Val e Teste)
print("\n Normalizando (StandardScaler)...")
scaler = StandardScaler()

# Identificar colunas de features (todas exceto as de controle)
cols_ignore = ['timestamp', 'activityID', 'subject_id']
feature_cols = [c for c in df_train.columns if c not in cols_ignore]

# Fit apenas no treino para evitar Data Leakage
df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])
df_val[feature_cols] = scaler.transform(df_val[feature_cols]) # NEW
df_test[feature_cols] = scaler.transform(df_test[feature_cols])

# 4. Segmentação
print(f"\n Segmentando janelas (Size={WINDOW_SIZE}, Step={STEP_SIZE})...")
X_train, y_train = generate_windows(df_train, WINDOW_SIZE, STEP_SIZE)
X_val, y_val = generate_windows(df_val, WINDOW_SIZE, STEP_SIZE) # NEW
X_test, y_test = generate_windows(df_test, WINDOW_SIZE, STEP_SIZE)

print(f"   Janelas Treino: {X_train.shape}")
print(f"   Janelas Validação: {X_val.shape}")
print(f"   Janelas Teste:  {X_test.shape}")

 Carregando arquivos brutos...
   Lendo subject101.dat...
   Lendo subject102.dat...
   Lendo subject103.dat...
   Lendo subject104.dat...
   Lendo subject105.dat...
   Lendo subject106.dat...
   Lendo subject107.dat...
   Lendo subject108.dat...
   Lendo subject109.dat...

 Limpando dados...
 Dados carregados. Shape Total: (1942868, 43)

 Dividindo Treino, Validação e Teste (Subject Independent)...
   Amostras Treino: 1151837
   Amostras Validação: 262102
   Amostras Teste:  522538

 Normalizando (StandardScaler)...

 Segmentando janelas (Size=250, Step=125)...
   Janelas Treino: (9136, 250, 40)
   Janelas Validação: (2080, 250, 40)
   Janelas Teste:  (4144, 250, 40)


In [4]:
# 5. Otimização de Memória e Encoding
print("\n Otimizando e Salvando...")

# Converter para float32 (reduz tamanho pela metade)
X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
X_test = X_test.astype(np.float32)

# Encoding dos Labels (Garante que todas as classes sejam aprendidas)
le = LabelEncoder()
all_labels = np.concatenate([y_train, y_val, y_test])
le.fit(all_labels)

y_train_enc = le.transform(y_train)
y_val_enc = le.transform(y_val)
y_test_enc = le.transform(y_test)

# Salvar .npz
np.savez_compressed(
    OUTPUT_PATH,
    X_train=X_train,
    y_train=y_train_enc,
    X_val=X_val,
    y_val=y_val_enc,
    X_test=X_test,
    y_test=y_test_enc,
    classes=le.classes_
)

print(f"Arquivo final salvo em: {OUTPUT_PATH}")
print("   Pode prosseguir para os experimentos!")


 Otimizando e Salvando...
Arquivo final salvo em: /content/drive/MyDrive/2025/Estudos/Datasets/pamap2/pamap2_v2.npz
   Pode prosseguir para os experimentos!
